# 49 -- Month-End Close Orchestrator

DAG-ordered task orchestration for a month-end financial close. Close tasks form a directed acyclic graph -- subledgers close before intercompany elimination, elimination and accruals feed consolidation, consolidation feeds reporting. The orchestrator tracks a typed `CloseTaskStatus` per task (`pending` / `in_progress` / `blocked` / `completed` / `exception`), only marks a task ready once every dependency is `completed`, and re-derives the critical path from scratch every time a status changes.

## Framework Comparison

DAG-ordered dependency gating is a well-known pattern outside agent frameworks too:

| Framework / tool | Equivalent construct | Key difference vs this example |
|---|---|---|
| **LangGraph** (used here) | `StateGraph` with a deterministic node computing readiness/critical-path, one LLM node for narrative | LangGraph here is just wiring -- almost all the logic is plain Python graph math, not agent reasoning |
| **Apache Airflow / Dagster** | DAG of tasks with `depends_on` / asset dependencies, scheduler dispatches ready tasks | Airflow/Dagster actually execute the tasks; this example only tracks and reports status supplied by the caller |
| **Critical Path Method (CPM) scheduling** | Forward/backward pass computing earliest start, float, and the critical path | Identical math to `compute_critical_path` -- longest path through a weighted DAG -- applied to accounting close tasks instead of a construction schedule |
| **CrewAI hierarchical process** | Manager agent that only delegates a subtask once its prerequisites are met | CrewAI's manager reasons about delegation with an LLM; here dependency gating is deterministic, not inferred |

In [ ]:
!pip install -q langgraph langchain-openai langchain-core pydantic python-dotenv

In [ ]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

## 1. The schema -- CloseTask, BlockedTaskInfo, CriticalPathInfo, CloseStatus

In [ ]:
# ruff: noqa: E402
"""
Pydantic models for the month-end close orchestrator.

CloseTask is the atomic unit of work in the close cycle DAG -- it carries its
own dependency list (depends_on) so the orchestrator can compute readiness
and critical path without a separate adjacency structure. BlockedTaskInfo and
CriticalPathInfo are the two derived-state objects the orchestrator recomputes
after every status change. CloseStatus is the live aggregate object returned
to the caller: overall progress, SLA risk, and the current critical path.
"""

from typing import Literal, Optional

from pydantic import BaseModel, Field

CloseTaskStatus = Literal["pending", "in_progress", "blocked", "completed", "exception"]

CloseTaskCategory = Literal[
    "subledger_close",
    "intercompany_elimination",
    "accruals_posting",
    "fx_revaluation",
    "consolidation",
    "reporting",
]

SLARisk = Literal["on_track", "at_risk", "critical"]


class CloseTask(BaseModel):
    task_id: str = Field(description="Unique identifier for this close task, e.g. 'T15'.")
    name: str = Field(description="Human-readable task name.")
    category: CloseTaskCategory = Field(description="Which stage of the close this task belongs to.")
    owner: str = Field(description="Team or role responsible for executing this task.")
    depends_on: list[str] = Field(
        default_factory=list,
        description="task_id values that must reach status='completed' before this task can start.",
    )
    sla_hours: float = Field(
        description="Hours this task is budgeted to take once it starts.", gt=0
    )
    status: CloseTaskStatus = Field(description="Current status of this task.")
    blocked_reason: Optional[str] = Field(
        default=None,
        description="Why this task is blocked or in exception. Required when status is 'blocked' or 'exception'.",
    )


class BlockedTaskInfo(BaseModel):
    task_id: str = Field(description="task_id of the blocked or exception task.")
    name: str = Field(description="Human-readable task name, for display without a second lookup.")
    reason: str = Field(description="Why the task cannot proceed.")
    estimated_delay_hours: float = Field(
        description="Estimated hours the close will slip if this task is not unblocked, ge=0."
    )
    downstream_impact: list[str] = Field(
        default_factory=list,
        description="task_id values of not-yet-completed tasks that transitively depend on this task.",
    )


class CriticalPathInfo(BaseModel):
    path: list[str] = Field(
        default_factory=list,
        description="task_id values in order, the longest remaining chain of incomplete work to close.",
    )
    total_remaining_hours: float = Field(
        description="Sum of sla_hours across every task on the critical path, ge=0."
    )
    bottleneck_task_id: Optional[str] = Field(
        default=None,
        description=(
            "task_id of the blocked or exception task sitting on the critical path, if any. "
            "None means the critical path is not currently obstructed."
        ),
    )


class CloseStatus(BaseModel):
    period: str = Field(description="Close period this status applies to, e.g. '2026-06'.")
    total_tasks: int = Field(description="Total number of tasks in the close task list.")
    completed_tasks: int = Field(description="Number of tasks with status='completed'.")
    overall_progress_pct: float = Field(
        description="completed_tasks / total_tasks as a percentage, 0-100.", ge=0.0, le=100.0
    )
    ready_tasks: list[str] = Field(
        default_factory=list,
        description="task_id values that are pending with all dependencies completed -- next to dispatch.",
    )
    blocked_tasks: list[BlockedTaskInfo] = Field(
        default_factory=list, description="Every task currently blocked or in exception."
    )
    critical_path: CriticalPathInfo = Field(
        description="The current longest remaining chain of incomplete work, re-derived on every call."
    )
    sla_risk: SLARisk = Field(
        description=(
            "'critical' if a blocked/exception task sits on the critical path, 'at_risk' if any "
            "task is blocked but off the critical path, otherwise 'on_track'."
        )
    )
    narrative: Optional[str] = Field(
        default=None,
        description="Executive narrative summarizing progress, SLA risk, and recommended action.",
    )

## 2. Deterministic DAG math -- readiness, blocked tasks, critical path (no LLM)

In [ ]:
# ruff: noqa: E402
"""
Deterministic DAG algorithms for the month-end close -- no LLM.

Every number the orchestrator reports (readiness, critical path, blocked-task
delay) is computed here in plain Python. The LLM is only handed the
already-computed figures and asked to narrate them, never to re-derive them.
"""


def topological_order(tasks: list[CloseTask]) -> list[str]:
    """Kahn's algorithm. Raises ValueError if the task list is not a DAG."""
    by_id = {t.task_id: t for t in tasks}
    in_degree = {t.task_id: 0 for t in tasks}
    children: dict[str, list[str]] = {t.task_id: [] for t in tasks}

    for task in tasks:
        for dep in task.depends_on:
            if dep not in by_id:
                raise ValueError(f"{task.task_id} depends on unknown task {dep!r}")
            children[dep].append(task.task_id)
            in_degree[task.task_id] += 1

    queue = sorted(task_id for task_id, deg in in_degree.items() if deg == 0)
    order: list[str] = []

    while queue:
        queue.sort()
        current = queue.pop(0)
        order.append(current)
        for child in children[current]:
            in_degree[child] -= 1
            if in_degree[child] == 0:
                queue.append(child)

    if len(order) != len(tasks):
        raise ValueError("Close task list contains a dependency cycle")

    return order


def compute_ready_tasks(tasks: list[CloseTask]) -> list[str]:
    """A task is ready to dispatch when it is pending and every dependency is completed."""
    by_id = {t.task_id: t for t in tasks}
    ready = []
    for task in tasks:
        if task.status != "pending":
            continue
        if all(by_id[dep].status == "completed" for dep in task.depends_on):
            ready.append(task.task_id)
    return sorted(ready)


def _descendants(task_id, children, by_id):
    """Every not-yet-completed task reachable downstream of task_id."""
    seen = set()
    stack = list(children.get(task_id, []))
    while stack:
        node = stack.pop()
        if node in seen:
            continue
        seen.add(node)
        stack.extend(children.get(node, []))
    return sorted(node for node in seen if by_id[node].status != "completed")


def compute_blocked_tasks(tasks: list[CloseTask]) -> list[BlockedTaskInfo]:
    """Surface every blocked/exception task with a reason and estimated delay."""
    by_id = {t.task_id: t for t in tasks}
    children: dict[str, list[str]] = {t.task_id: [] for t in tasks}
    for task in tasks:
        for dep in task.depends_on:
            children[dep].append(task.task_id)

    blocked = []
    for task in tasks:
        if task.status not in ("blocked", "exception"):
            continue
        blocked.append(
            BlockedTaskInfo(
                task_id=task.task_id,
                name=task.name,
                reason=task.blocked_reason or "No reason recorded.",
                estimated_delay_hours=task.sla_hours,
                downstream_impact=_descendants(task.task_id, children, by_id),
            )
        )
    return sorted(blocked, key=lambda b: b.task_id)


def compute_critical_path(tasks: list[CloseTask]) -> CriticalPathInfo:
    """Longest remaining chain of incomplete work through the DAG."""
    order = topological_order(tasks)
    by_id = {t.task_id: t for t in tasks}

    dist: dict[str, float] = {}
    prev: dict = {}

    for task_id in order:
        task = by_id[task_id]

        if task.status == "completed":
            dist[task_id] = 0.0
            prev[task_id] = None
            continue

        best_dep_dist = 0.0
        best_dep = None
        for dep in task.depends_on:
            dep_dist = dist.get(dep, 0.0)
            if dep_dist >= best_dep_dist:
                best_dep_dist = dep_dist
                best_dep = dep

        dist[task_id] = task.sla_hours + best_dep_dist
        prev[task_id] = best_dep if best_dep_dist > 0 else None

    incomplete_ids = [t.task_id for t in tasks if t.status != "completed"]
    if not incomplete_ids:
        return CriticalPathInfo(path=[], total_remaining_hours=0.0, bottleneck_task_id=None)

    sink = max(incomplete_ids, key=lambda tid: dist[tid])

    path = []
    node = sink
    while node is not None:
        path.append(node)
        node = prev[node]
    path.reverse()

    bottleneck = next(
        (tid for tid in path if by_id[tid].status in ("blocked", "exception")), None
    )

    return CriticalPathInfo(
        path=path,
        total_remaining_hours=round(dist[sink], 2),
        bottleneck_task_id=bottleneck,
    )

## 3. The one LLM call -- narrative synthesis

In [ ]:
# ruff: noqa: E402
"""The close-status narrative synthesizer -- the only LLM call in this example.

Readiness, blocked-task delay, and critical path are all deterministic graph
math computed above. This is the one genuinely open-ended writing task --
turning a typed CloseStatus snapshot into an executive narrative.
"""

import json

from langchain_openai import ChatOpenAI

_MODEL = "gpt-4.1-nano"

CLOSE_NARRATIVE_SYSTEM = (
    "You are a month-end close controller writing a status update for the accounting "
    "leadership team. You will receive a JSON payload with: period, total_tasks, "
    "completed_tasks, overall_progress_pct, ready_tasks, blocked_tasks (each with reason, "
    "estimated_delay_hours, downstream_impact), critical_path (path, total_remaining_hours, "
    "bottleneck_task_id), and sla_risk.\n\n"
    "Write a 3-5 sentence executive narrative that:\n"
    "  1. States overall progress using the exact overall_progress_pct and completed_tasks/"
    "total_tasks figures provided -- never estimate or round differently than given.\n"
    "  2. If blocked_tasks is non-empty, names the specific blocked task(s), the reason "
    "given, and how many downstream tasks are impacted.\n"
    "  3. If critical_path.bottleneck_task_id is set, calls out that this blocked task is "
    "on the critical path and is therefore delaying the entire close by "
    "critical_path.total_remaining_hours hours -- use sla_risk to set the tone (critical "
    "means escalate now, at_risk means monitor, on_track means no action needed).\n"
    "  4. Ends with one concrete recommended next action for the controller.\n\n"
    "Never invent a task, owner, or number that is not present in the payload. "
    "Return plain text, no markdown headers."
)


def close_narrative_synthesizer(close_status_payload: dict) -> str:
    """close_status_payload is a CloseStatus.model_dump() with narrative=None."""
    llm = ChatOpenAI(model=_MODEL, temperature=0)
    response = llm.invoke(
        [("system", CLOSE_NARRATIVE_SYSTEM), ("human", json.dumps(close_status_payload))]
    )
    return response.content

## 4. The graph -- compute_status -> synthesize

In [ ]:
# ruff: noqa: E402
from typing import TypedDict

from langgraph.graph import END, START, StateGraph


class GraphState(TypedDict):
    period: str
    tasks: list[dict]
    status: dict


def _compute_status(state: GraphState) -> dict:
    tasks = [CloseTask.model_validate(t) for t in state["tasks"]]

    total_tasks = len(tasks)
    completed_tasks = sum(1 for t in tasks if t.status == "completed")
    overall_progress_pct = round((completed_tasks / total_tasks) * 100, 2) if total_tasks else 0.0

    ready_tasks = compute_ready_tasks(tasks)
    blocked_tasks = compute_blocked_tasks(tasks)
    critical_path = compute_critical_path(tasks)

    if critical_path.bottleneck_task_id is not None:
        sla_risk = "critical"
    elif blocked_tasks:
        sla_risk = "at_risk"
    else:
        sla_risk = "on_track"

    status = CloseStatus(
        period=state["period"],
        total_tasks=total_tasks,
        completed_tasks=completed_tasks,
        overall_progress_pct=overall_progress_pct,
        ready_tasks=ready_tasks,
        blocked_tasks=blocked_tasks,
        critical_path=critical_path,
        sla_risk=sla_risk,
        narrative=None,
    )
    return {"status": status.model_dump()}


def _synthesize(state: GraphState) -> dict:
    narrative = close_narrative_synthesizer(state["status"])
    status = dict(state["status"])
    status["narrative"] = narrative
    return {"status": status}


graph = StateGraph(GraphState)
graph.add_node("compute_status", _compute_status)
graph.add_node("synthesize", _synthesize)
graph.add_edge(START, "compute_status")
graph.add_edge("compute_status", "synthesize")
graph.add_edge("synthesize", END)
app = graph.compile()


def run(period: str, tasks: list[CloseTask]) -> CloseStatus:
    """Run one orchestration pass and return a live CloseStatus.

    Call again with an updated task list (e.g. after a blocker resolves) to
    re-derive readiness and critical path from scratch.
    """
    result = app.invoke({"period": period, "tasks": [t.model_dump() for t in tasks]})
    return CloseStatus.model_validate(result["status"])

## 5. Run it -- clean close vs blocked bottleneck

In [ ]:
# task_id -> (name, category, owner, depends_on, sla_hours)
_TASK_DEFS = {
    "T1": ("AP subledger close", "subledger_close", "AP Team", [], 4),
    "T2": ("AR subledger close", "subledger_close", "AR Team", [], 4),
    "T3": ("Fixed assets subledger close", "subledger_close", "Fixed Assets Team", [], 3),
    "T4": ("Inventory subledger close", "subledger_close", "Inventory Team", [], 5),
    "T5": ("IC transaction matching", "intercompany_elimination", "Intercompany Team", ["T1", "T2"], 6),
    "T6": ("IC elimination entries", "intercompany_elimination", "Intercompany Team", ["T5"], 4),
    "T7": ("IC out-of-balance investigation", "intercompany_elimination", "Intercompany Team", ["T5"], 3),
    "T8": ("Payroll accrual", "accruals_posting", "Payroll Team", ["T1"], 3),
    "T9": ("Bonus accrual true-up", "accruals_posting", "Payroll Team", ["T8"], 2),
    "T10": ("Vendor accrual true-up", "accruals_posting", "AP Team", ["T1"], 4),
    "T11": ("Revenue accrual", "accruals_posting", "AR Team", ["T2"], 3),
    "T12": ("FX rate table load", "fx_revaluation", "Treasury", [], 1),
    "T13": ("Balance sheet FX revaluation", "fx_revaluation", "Treasury", ["T12", "T1", "T2", "T3", "T4"], 5),
    "T14": ("Intercompany balance FX revaluation", "fx_revaluation", "Treasury", ["T12", "T6"], 3),
    "T15": ("Trial balance roll-up", "consolidation", "Consolidation Team", ["T7", "T8", "T9", "T10", "T11", "T13", "T14"], 6),
    "T16": ("Minority interest calculation", "consolidation", "Consolidation Team", ["T15"], 2),
    "T17": ("Consolidation adjustments", "consolidation", "Consolidation Team", ["T15"], 4),
    "T18": ("Elimination review sign-off", "consolidation", "Controller", ["T6", "T17"], 2),
    "T19": ("Draft financial statements", "reporting", "Reporting Team", ["T16", "T17", "T18"], 5),
    "T20": ("Management review", "reporting", "CFO", ["T19"], 3),
    "T21": ("Audit support package", "reporting", "Reporting Team", ["T19"], 4),
    "T22": ("Board reporting package", "reporting", "Reporting Team", ["T20"], 3),
    "T23": ("Regulatory filing prep", "reporting", "Compliance Team", ["T20"], 6),
    "T24": ("Final close sign-off", "reporting", "Controller", ["T21", "T22", "T23"], 1),
}

PERIOD = "2026-06"


def build_tasks(completed_ids, blocked=None):
    blocked = blocked or {}
    tasks = []
    for task_id, (name, category, owner, depends_on, sla_hours) in _TASK_DEFS.items():
        if task_id in blocked:
            status, reason = "blocked", blocked[task_id]
        elif task_id in completed_ids:
            status, reason = "completed", None
        else:
            status, reason = "pending", None
        tasks.append(
            CloseTask(
                task_id=task_id, name=name, category=category, owner=owner,
                depends_on=depends_on, sla_hours=sla_hours, status=status, blocked_reason=reason,
            )
        )
    return tasks


def print_status(label, status):
    print(f"\n{'=' * 70}\n{label}\n{'=' * 70}")
    print(f"Progress        : {status.completed_tasks}/{status.total_tasks} ({status.overall_progress_pct}%)")
    print(f"Ready to dispatch: {status.ready_tasks}")
    print(f"SLA risk        : {status.sla_risk}")
    print(f"Critical path   : {' -> '.join(status.critical_path.path)} ({status.critical_path.total_remaining_hours}h remaining)")
    if status.critical_path.bottleneck_task_id:
        print(f"Bottleneck      : {status.critical_path.bottleneck_task_id}")
    for b in status.blocked_tasks:
        print(f"  BLOCKED {b.task_id} ({b.name}): {b.reason}")
    print(f"\nNarrative:\n{status.narrative}")


# Scenario 1 -- clean close, nothing blocked
clean_completed = {f"T{n}" for n in range(1, 15)}
clean_status = run(PERIOD, build_tasks(clean_completed))
print_status("Scenario 1 -- Clean Close", clean_status)

In [ ]:
# Scenario 2 -- IC out-of-balance investigation (T7) blocked, gating the
# entire consolidation/reporting chain, then resolved and re-derived
blocked_completed = {"T1", "T2", "T3", "T4", "T5", "T6", "T8", "T9", "T10", "T11", "T12", "T13", "T14"}
blocked_reasons = {
    "T7": "Intercompany discrepancy of $18,750 between US and UK entities is unresolved -- awaiting bank confirmation from UK treasury."
}
blocked_status = run(PERIOD, build_tasks(blocked_completed, blocked_reasons))
print_status("Scenario 2 -- Blocked: IC Out-of-Balance Investigation", blocked_status)

resolved_status = run(PERIOD, build_tasks(blocked_completed | {"T7"}))
print_status("Scenario 2 (resolved) -- IC Investigation Cleared", resolved_status)

## Starter Exercise

**Detect SLA breaches on `in_progress` tasks, not just `blocked` ones.**

Right now `sla_risk` only reacts to tasks explicitly marked `blocked` or `exception`. But a task can quietly blow its SLA while still `in_progress` -- nobody flagged it, it is just taking longer than budgeted.

Add:
1. A `hours_elapsed: Optional[float]` field to `CloseTask` -- how many hours the task has actually been running.
2. A `compute_overdue_tasks(tasks)` function (same style as `compute_blocked_tasks`) that returns every `in_progress` task where `hours_elapsed > sla_hours`.
3. An update to `_compute_status`'s `sla_risk` logic so an overdue task **on the critical path** also triggers `'critical'`, even though its status is `in_progress`, not `blocked`.

Think through:
- Does `compute_critical_path` need to change, or does it already treat `in_progress` correctly (duration = `sla_hours` for any non-completed status)?
- Should an overdue-but-off-critical-path task raise `sla_risk` to `at_risk`, the same way a blocked-but-off-critical-path task does?
- What does `CLOSE_NARRATIVE_SYSTEM` need to say differently for an overdue task vs a blocked one?

Try writing the answer yourself before looking at the answer key below.

In [ ]:
# Your code here

### Answer Key

In [ ]:
from typing import Optional


class CloseTaskWithElapsed(CloseTask):
    hours_elapsed: Optional[float] = Field(
        default=None,
        description="Hours this in_progress task has actually been running -- used to detect a silent SLA breach.",
    )


def compute_overdue_tasks(tasks):
    """in_progress tasks that have already exceeded their sla_hours budget."""
    overdue = []
    for task in tasks:
        if task.status != "in_progress":
            continue
        elapsed = getattr(task, "hours_elapsed", None)
        if elapsed is not None and elapsed > task.sla_hours:
            overdue.append(task.task_id)
    return sorted(overdue)


# compute_critical_path already treats any non-completed status (including
# in_progress) as contributing its full sla_hours to the path -- no change
# needed there. The only change is in sla_risk classification:
#
# def _compute_status(state):
#     ...
#     overdue = compute_overdue_tasks(tasks)
#     overdue_on_path = any(tid in critical_path.path for tid in overdue)
#     if critical_path.bottleneck_task_id is not None or overdue_on_path:
#         sla_risk = "critical"
#     elif blocked_tasks or overdue:
#         sla_risk = "at_risk"
#     else:
#         sla_risk = "on_track"
#
# And CLOSE_NARRATIVE_SYSTEM gains a fourth payload field, overdue_tasks,
# with an instruction to name any in_progress task exceeding its SLA
# separately from blocked tasks -- it is a schedule risk, not a hard gate.

demo_tasks = build_tasks(clean_completed)
demo_tasks[14] = demo_tasks[14].model_copy(update={"status": "in_progress"})
print("Overdue (no hours_elapsed set, so none flagged):", compute_overdue_tasks(demo_tasks))